<a href="https://colab.research.google.com/github/yooniqx/yooniqx/blob/main/Copy_of_ADK_Learning_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Welcome to Your ADK Adventure - Tools & Memory! 🚀

Welcome, Agent Architect! This notebook is your guide to giving your AI agents two essential superpowers: custom tools and conversational memory.

By the end of this adventure, you will be able to:

- **Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

## Author

HI, I'm Qingyue (Annie) Wang, a developer advocate and AI engineer at **Google**, passionate about helping developers build with AI and cloud technologies :)


If you have questions with this notebook, contact me on [LinkedIn](https://www.linkedin.com/in/anniewangtech/) , [X](https://twitter.com/anniewangtech) or email anniewangtech0510@Gmail.com


```
  (\__/)
  (•ㅅ•)
  /づ  📚      Enjoy learning AI Agents :)
```


-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: https://codelabs.developers.google.com/onramp/instructions#1

 -----------------------------------------------------------------------------

```
 ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )             
```


## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [1]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
import vertexai
from google.colab import auth
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")


✅ All libraries are ready to go!


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


### Authenticate and Configure Your Project
To use Vertex AI, you need an active Google Cloud project. This section handles authenticating your environment and setting up the necessary project configurations.

In [2]:
# ---  Authentication & Project Configuration ---

# Authenticate user in Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()
    print("✅ Authenticated successfully.")

✅ Authenticated successfully.


In [3]:
# @title Set Your Google Cloud Project Details
PROJECT_ID = "black-seer-341703"             # @param {type:"string"}
LOCATION = "us-central1"               # @param {type:"string"}

# Set environment variables for the ADK and gcloud
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

print(f"\n✅ Vertex AI configured for project '{PROJECT_ID}' in '{LOCATION}'.")


✅ Vertex AI configured for project 'black-seer-341703' in 'us-central1'.


---
## Part 1: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

```
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [4]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [5]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [6]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: '8ae9c410-56a3-4ed2-a3a7-d84f891a4134'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's a relaxing and artsy, yet affordable, day trip itinerary near Sunnyvale, CA, perfect for a Saturday adventure!

## Spontaneous Day Trip: Arts & Roses in the South Bay

This itinerary blends world-class art, fragrant gardens, and cultural exploration without breaking the bank, focusing on free attractions and affordable dining in Palo Alto and San Jose.

---

### 9:30 AM - Morning: Artistic Immersion at Stanford University (Palo Alto)

Start your day with a visit to the renowned **Cantor Arts Center at Stanford University** in Palo Alto. This incredible museum offers free admission to its extensive collection, spanning 5,000 years of art from around the globe. You can wander through galleries showcasing

Here's a relaxing and artsy, yet affordable, day trip itinerary near Sunnyvale, CA, perfect for a Saturday adventure!

## Spontaneous Day Trip: Arts & Roses in the South Bay

This itinerary blends world-class art, fragrant gardens, and cultural exploration without breaking the bank, focusing on free attractions and affordable dining in Palo Alto and San Jose.

---

### 9:30 AM - Morning: Artistic Immersion at Stanford University (Palo Alto)

Start your day with a visit to the renowned **Cantor Arts Center at Stanford University** in Palo Alto. This incredible museum offers free admission to its extensive collection, spanning 5,000 years of art from around the globe. You can wander through galleries showcasing everything from ancient artifacts to contemporary works. Afterwards, step outside into the **Rodin Sculpture Garden**, which is open 24/7 and houses one of the largest collections of Rodin bronzes outside of Paris, including his famous "The Thinker." The beautiful Stanford campus itself is perfect for a relaxing stroll, offering a serene atmosphere. Parking on the Stanford campus is free on weekends.

*   **Location:** Cantor Arts Center, 328 Lomita Dr, Stanford, CA 94305
*   **Operating Hours (Saturday):** Cantor Arts Center: 10:00 AM - 5:00 PM; Rodin Sculpture Garden: 24 hours
*   **Cost:** Free admission to both the museum and sculpture garden.

### 1:00 PM - Lunch & Cultural Exploration: San Jose Japantown

Head south to **San Jose Japantown**, one of the last three historic Japantowns in the United States. This vibrant neighborhood offers a unique blend of eclectic boutiques, traditional shops, art galleries, and murals. For an affordable and authentic lunch, explore the many Japanese eateries. Options like **Minato Japanese Restaurant** or **Kumako Ramen** offer delicious and reasonably priced meals, with lunch specials often available. **Kaita Restaurant** is also known for its curry and comfort food. Take some time to browse the shops and soak in the cultural atmosphere. Street parking is available, typically around $2 per hour.

*   **Location:** Jackson Street, San Jose, CA 95112 (center around N 5th and Jackson Streets)
*   **Cost:** Affordable lunch options (prices vary by restaurant).

### 3:00 PM - Afternoon: Serenity at the San Jose Municipal Rose Garden

Just a short drive from Japantown, immerse yourself in the beauty and tranquility of the **San Jose Municipal Rose Garden**. Voted "America's Best Rose Garden," this 5.5-acre park boasts over 3,500 rose bushes and 189 varieties, with peak bloom usually occurring from April through November. It's a perfect spot for a relaxing walk, enjoying the fragrant blooms and picturesque scenery. There is no entrance fee, and the garden is open from 8 a.m. to an hour after sunset, every day of the year. Free public parking is also available.

*   **Location:** 1649 Naglee Ave, San Jose, CA 95126
*   **Operating Hours:** 8:00 AM - one hour after sunset.
*   **Cost:** Free admission.

### 6:30 PM - Evening: Dinner & Evening Stroll in Japantown

Return to **San Jose Japantown** for a casual and affordable dinner. You can revisit one of the lunch spots you liked, or try another establishment like **Gombei** for traditional Japanese fare, or **Kaita Restaurant** for comfort food. After dinner, enjoy a leisurely evening stroll through the illuminated streets of Japantown, perhaps stopping for a boba tea or a sweet treat. The unique shops and vibrant atmosphere make for a pleasant end to your artsy and relaxing day.

*   **Location:** Jackson Street, San Jose, CA 95112
*   **Cost:** Affordable dinner options (prices vary by restaurant).

--------------------------------------------------



---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [7]:
# --- Tool Definition: A function that calls a live public API ---
import requests
import json

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [8]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "I want to go hiking near Lake Tahoe, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'I want to go hiking near Lake Tahoe, what's the weather like?'

🚀 Running query for agent: 'weather_aware_planner' in session: 'c3f07a7f-b838-4302-aac2-7234a882d9a6'...


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'location': 'Lake Tahoe'
        },
        id='adk-7c5746c8-3dc8-4bc5-b6ed-6ba9abca299a',
        name='get_live_weather_forecast'
      ),
      thought_signature=b'\n\xcb\x02\x01\x8f=k_^\x04\x17HY|\xfa\x0f<\xe9\xd3\xa6\xbdq\xbd\x05}M\xf1\x8e\xcfK\x9c$R0\xb1\x90L6\xa3\xd2\t1Q2\xc5e\xa3\x1f^\xb8\x96+Bm\x1f>\xd06 \xceA\x0e\x93\x83\xc7\xb8\x92\xba\\E~\xd2\xa8\x9dF;\xc3S\xd0F\x80To\xcc\xc7\xeb\x8f\xe1\xc8o\xc0\x8c\xacE\x0e}\xfd...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=10,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=10
    ),
  ],
  prompt_token_c

The weather near Lake Tahoe is partly cloudy with a low around 42°F and a light southwest wind of 0 to 5 mph. It's a bit chilly, so if you plan on hiking, make sure to dress in layers!

--------------------------------------------------



## 2.2 The Agent-as-a-Tool: Consulting a Specialist 🧑‍🍳

Why build one agent that does everything when you can build a **team of specialist agents?** The **Agent-as-a-Tool** pattern allows one agent to delegate a task to another agent.

**Key Concept:** This is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed **back to Agent A**. Agent A then uses that information to form its own final response to the user. It's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.

### How It Works

Our top-level agent, the `trip_data_concierge_agent`, acts as the **Orchestrator**. It has two tools at its disposal:

1.  `call_db_agent`: A function that internally calls our `db_agent` to fetch raw data.
2.  `call_concierge_agent`: A function that calls the `concierge_agent`.

The `concierge_agent` itself has a tool: the `food_critic_agent`.

The flow for a complex query is:

1.  **User** asks the `trip_data_concierge_agent` for a hotel and a nearby restaurant.
2.  The **Orchestrator** first calls `call_db_agent` to get hotel data.
3.  The data is saved in `tool_context.state`.
4.  The **Orchestrator** then calls `call_concierge_agent`, which retrieves the hotel data from the context.
5.  The `concierge_agent` receives the request and decides it needs to use its own tool, the `food_critic_agent`.
6.  The `food_critic_agent` provides a witty recommendation.
7.  The `concierge_agent` gets the critic's response and politely formats it.
8.  This final, polished response is returned to the **Orchestrator**, which presents it to the user.

                         +-----------------------------------------------------------+
                         |              🧭 Trip Data Concierge Agent                 |
                         |-----------------------------------------------------------|
                         |  Model: gemini-2.5-flash                                  |
                         |  Description:                                             |
                         |   Orchestrates database query and travel recommendation  |
                         |-----------------------------------------------------------|
                         |  🔧 Tools:                                                |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |            🔧 Tool: call_db_agent         |    |         🔧 Tool: call_concierge_agent        |
        |-------------------------------------------|    |---------------------------------------------|
        | Calls: db_agent                           |    | Calls: concierge_agent                       |
        |                                           |    | Uses data from db_agent for recommendations |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | Model: gemini-2.5-flash                    |   | Model: gemini-2.5-flash                         |
       | Role: Return mock JSON hotel data          |   | Role: Hotel staff that handles user Q&A        |
       +--------------------------------------------+   | Tools:                                          |
                                                         |  - food_critic_agent                           |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | Model: gemini-2.5-flash                         |
                                                       | Role: Gives a witty restaurant recommendation   |
                                                       +------------------------------------------------+


In [9]:
import asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool

# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents.

db_agent = Agent(
    name="db_agent",
    model="gemini-2.5-flash",
    instruction="You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}")

# --- 1. Define the Specialist Agents ---

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name="food_critic_agent",
    model="gemini-2.5-flash",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
)

# The Concierge knows how to use the Food Critic
concierge_agent = Agent(
    name="concierge_agent",
    model="gemini-2.5-flash",
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely.",
    tools=[AgentTool(agent=food_critic_agent)]
)


# --- 2. Define the Tools for the Orchestrator ---

async def call_db_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # Store the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieve the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # Formulate a new prompt for the concierge, giving it the data context
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# --- 3. Define the Top-Level Orchestrator Agent ---

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model="gemini-2.5-flash",
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

✅ Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [10]:
# --- Let's test the Trip Data Concierge Agent ---

async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Create a new, single-use session for this query
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # This query is specifically designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    print(f"🗣️ User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

🗣️ User Query: 'Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews.'

🚀 Running query for agent: 'trip_data_concierge' in session: 'f626f081-c5f3-4f52-8a1e-4aad2ccda881'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'top-rated hotels in San Francisco'
        },
        id='adk-e4912508-9c48-4476-8639-4ffe4a0a5ffa',
        name='call_db_agent'
      ),
      thought_signature=b'\n\xdb\x03\x01\x8f=k_n9P\xbe(\xbfF2\xe8\x89\x9cWP\xb2\xf1`\xc4\xb2Mz\xd4\x85\x840*\xe5&@\x05\x0b7\xc6_\xd9\x11\xe7\xf7\xf4\x95\x90\x804\xed\xfe\x0e\xc1\xbc\x8f\x98\xb3;\xce\xdcB\xa8\xf5\x934$\x9f\\\\\xf0\xaf\x1c\x82\xdc\xf5\x08\x99\x05c\xad\xf6h\x08\x02\xa0\x94_\x1b\x83\x05.V\x1f:\xbc ...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_

I'm sorry, but I cannot fulfill your request for real-time top-rated hotels in San Francisco. My `call_db_agent` tool indicates that I do not have access to a database that provides current hotel information and can only provide mock data.

Therefore, I cannot identify the hotel with the most reviews or suggest a dinner spot near it.

Would you like me to try with some mock data, or is there another type of travel-related question I can help you with?

--------------------------------------------------



---
## Part 3: Agent with a Memory - The Adaptive Planner 🗺️

Now, let's see an agent that not only **remembers** but also **adapts**. We'll challenge the `multi_day_trip_agent` to re-plan part of its itinerary based on our feedback. This is a much more realistic test of conversational AI.

```
+-----------------------------------------------------+
|         Adaptive Multi-Day Trip Agent 🗺️           |
|-----------------------------------------------------|
|  Model: gemini-2.5-flash                            |
|  Description:                                       |
|   Builds multi-day travel itineraries step-by-step, |
|   remembers previous days, adapts to feedback       |
|-----------------------------------------------------|
|  🔧 Tools:                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capabilities:                                   |
|   - Memory of past conversation & preferences       |
|   - Progressive planning (1 day at a time)          |
|   - Adapts to user feedback                         |
|   - Ensures activity variety across days            |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     User Interaction      |
    |---------------------------|
    | - Destination             |
    | - Trip duration           |
    | - Interests & feedback    |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Day-by-Day Itinerary Generation              |
|-----------------------------------------------------|
|  🗓️ Day N Output (Markdown format):                 |
|   - Morning / Afternoon / Evening activities        |
|   - Personalized & context-aware                    |
|   - Changes accepted, feedback acknowledged         |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Next Day Planning Triggered 🚀               |
|-----------------------------------------------------|
| - Builds on prior days                              |
| - Avoids repetition                                 |
| - Asks user for confirmation before proceeding      |
+-----------------------------------------------------+
```


In [11]:
# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### Scenario 3a: Agent WITH Memory (Using a SINGLE Session) ✅

First, let's see the correct way to do it. We will use the **exact same `trip_session` object** for the entire conversation. Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.

In [12]:
# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # --- Turn 1: The user initiates the trip ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: The user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: The user confirms and asks to continue ---
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: 712690b1-43d3-41e4-9472-221b5c75e578

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '712690b1-43d3-41e4-9472-221b5c75e578'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Okay, fantastic! Lisbon is a wonderful choice with its rich history and delicious food. I'll help you plan a memorable 2-day trip.

Let's start with **Day 1**. How does this sound?

### Day 1: Historic Alfama & Culinary Delights

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Begin your day in the oldest district of Lisbon, Alfama. Wander through its narrow, winding streets, soak in the medieval atmosphere, and discover hidden courtyards.
    *   Visit the **Lisbon Cathedral (Sé de Lisboa)**, a magnificent Romanesque cat

Okay, fantastic! Lisbon is a wonderful choice with its rich history and delicious food. I'll help you plan a memorable 2-day trip.

Let's start with **Day 1**. How does this sound?

### Day 1: Historic Alfama & Culinary Delights

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Begin your day in the oldest district of Lisbon, Alfama. Wander through its narrow, winding streets, soak in the medieval atmosphere, and discover hidden courtyards.
    *   Visit the **Lisbon Cathedral (Sé de Lisboa)**, a magnificent Romanesque cathedral dating back to the 12th century.
    *   Climb up to the **Miradouro das Portas do Sol** and **Miradouro de Santa Luzia** for breathtaking panoramic views of the Alfama rooftops and the Tagus River.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Cuisine**
    *   Enjoy a traditional Portuguese lunch in Alfama. Look for a local *tasca* (a small, informal restaurant) to try dishes like *Bacalhau à Brás* (codfish with scrambled eggs, potatoes, and onions) or grilled sardines (seasonal).
*   **Afternoon (2:30 PM - 5:30 PM): São Jorge Castle**
    *   Head to the iconic **São Jorge Castle (Castelo de São Jorge)**. Explore its ancient walls, archaeological site, and peacock-filled gardens while enjoying unparalleled views over Lisbon.
*   **Evening (7:00 PM onwards): Fado Show with Dinner**
    *   Experience an authentic Portuguese Fado show in Alfama or the nearby Mouraria district. Many Fado houses offer dinner alongside the performance, giving you a taste of local culture and music.

What do you think of this plan for Day 1?

--------------------------------------------------


🗣️ User (Turn 2 - Feedback): 'That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '712690b1-43d3-41e4-9472-221b5c75e578'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""You're right, São Jorge Castle is indeed a castle! My apologies for that oversight. I understand you'd prefer something else for the historical afternoon activity.

Let's revise **Day 1** to incorporate a different historical experience while keeping your interest in local food and the Alfama exploration.

Here's an updated plan for Day 1:

### Day 1: Historic Alfama & Belém's Age of Discovery

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Begin your day in the oldest district of Lisbon, Alfama. Wander through its narrow, winding streets, soak in the m

You're right, São Jorge Castle is indeed a castle! My apologies for that oversight. I understand you'd prefer something else for the historical afternoon activity.

Let's revise **Day 1** to incorporate a different historical experience while keeping your interest in local food and the Alfama exploration.

Here's an updated plan for Day 1:

### Day 1: Historic Alfama & Belém's Age of Discovery

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Begin your day in the oldest district of Lisbon, Alfama. Wander through its narrow, winding streets, soak in the medieval atmosphere, and discover hidden courtyards.
    *   Visit the **Lisbon Cathedral (Sé de Lisboa)**, a magnificent Romanesque cathedral dating back to the 12th century.
    *   Climb up to the **Miradouro das Portas do Sol** and **Miradouro de Santa Luzia** for breathtaking panoramic views of the Alfama rooftops and the Tagus River.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Cuisine**
    *   Enjoy a traditional Portuguese lunch in Alfama. Look for a local *tasca* (a small, informal restaurant) to try dishes like *Bacalhau à Brás* (codfish with scrambled eggs, potatoes, and onions) or grilled sardines (seasonal).
*   **Afternoon (2:30 PM - 5:30 PM): Belém's Age of Discoveries**
    *   Head to the historic Belém district, a UNESCO World Heritage site, famous for its connections to Portugal's Age of Discoveries.
    *   Visit the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, an impressive example of Manueline architecture and the resting place of Vasco da Gama. Explore its stunning church and cloisters.
    *   See the iconic **Belém Tower (Torre de Belém)**, a fortified tower that once guarded the entrance to Lisbon's harbor.
    *   Walk by the grand **Monument to the Discoveries (Padrão dos Descobrimentos)**, celebrating Portugal's explorers.
    *   *Optional Treat:* Don't forget to try a world-famous *Pastel de Nata* from the original Pastéis de Belém bakery!
*   **Evening (7:00 PM onwards): Fado Show with Dinner**
    *   Experience an authentic Portuguese Fado show in Alfama or the nearby Mouraria district. Many Fado houses offer dinner alongside the performance, giving you a taste of local culture and music.

How does this revised Day 1 plan look to you?

--------------------------------------------------


🗣️ User (Turn 3 - Confirmation): 'Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '712690b1-43d3-41e4-9472-221b5c75e578'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Great! I'm glad Day 1 is to your liking. Now, let's plan a delicious and historically rich **Day 2** for your trip to Lisbon, keeping that fantastic food theme in mind.

### Day 2: Baixa, Chiado, and Culinary Exploration

*   **Morning (9:00 AM - 1:00 PM): Baixa & Chiado Districts - Urban History & Grandeur**
    *   Begin your day in the **Baixa** district, rebuilt after the 1755 earthquake with a unique grid-like street plan. Stroll through the grand **Praça do Comércio**, one of Europe's largest and most beautiful squares, opening onto the Tagus River.
    *   Walk along **Rua Augusta**, a bustling pedestrian s

Great! I'm glad Day 1 is to your liking. Now, let's plan a delicious and historically rich **Day 2** for your trip to Lisbon, keeping that fantastic food theme in mind.

### Day 2: Baixa, Chiado, and Culinary Exploration

*   **Morning (9:00 AM - 1:00 PM): Baixa & Chiado Districts - Urban History & Grandeur**
    *   Begin your day in the **Baixa** district, rebuilt after the 1755 earthquake with a unique grid-like street plan. Stroll through the grand **Praça do Comércio**, one of Europe's largest and most beautiful squares, opening onto the Tagus River.
    *   Walk along **Rua Augusta**, a bustling pedestrian street, and pass under its triumphal arch.
    *   Take the historic **Santa Justa Lift** (or one of the funiculars like Elevador da Glória or Bica) up to the **Chiado** district. Enjoy the elegant streets, historic theaters, and charming cafes of this sophisticated area.
    *   Visit the atmospheric ruins of the **Carmo Convent (Convento do Carmo)**, a gothic church destroyed by the 1755 earthquake, which now houses an archaeological museum.
*   **Lunch (1:00 PM - 2:30 PM): Time Out Market Lisbon (Mercado da Ribeira)**
    *   Immerse yourself in Lisbon's vibrant food scene at the **Time Out Market Lisbon (Mercado da Ribeira)**. This historic market has been transformed into a fantastic food hall, featuring stalls from some of the city's best chefs and restaurants, offering a wide array of traditional Portuguese dishes and contemporary cuisine. It's a perfect spot to try various local specialties.
*   **Afternoon (2:30 PM - 5:30 PM): Bairro Alto Views & Gourmet Exploration**
    *   After lunch, explore the area around the market. Take a leisurely walk through some of the smaller streets or enjoy a coffee at a local cafe.
    *   Head up to the **Miradouro de São Pedro de Alcântara** in Bairro Alto for another stunning panoramic view, this time overlooking the Baixa district and São Jorge Castle.
    *   Consider a short **Gourmet Food Walking Tour** (pre-booking recommended) in the afternoon, if you're keen to delve deeper into specific Portuguese delicacies, pastries, or wines with a local expert. This could take you to specialty shops, bakeries, and wine bars.
*   **Evening (7:00 PM onwards): Dinner in Principe Real or Bairro Alto**
    *   Enjoy your final dinner in Lisbon in the trendy **Principe Real** or lively **Bairro Alto** districts. Both offer an excellent selection of restaurants, from traditional Portuguese to more modern and innovative cuisine. Seek out a restaurant specializing in fresh seafood, or perhaps try a modern take on classic Portuguese dishes. Afterward, if you wish, you can experience the vibrant nightlife of Bairro Alto.

How does this itinerary for Day 2 sound for concluding your Lisbon adventure?

--------------------------------------------------



### Scenario 3b: Agent WITHOUT Memory (Using SEPARATE Sessions) ❌

Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a **brand new, separate session for each turn**.

Pay close attention to the agent's response to the second query. Because it's in a new session, it has no memory of the trip to Lisbon we just discussed!

In [13]:
# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()


############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: 6188a860-8252-4b37-a0ac-e39f628cbbd8
🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '6188a860-8252-4b37-a0ac-e39f628cbbd8'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Great! Lisbon is a fantastic choice with its rich history and delicious food. I'll help you plan a wonderful 2-day trip.

Let's start with **Day 1**. How does this sound?

### Day 1: Historic Alfama & Baixa

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Begin your day in Alfama, one of Lisbon's oldest and most charming neighborhoods. Wander through its narrow, winding streets, soak in the atmosphere, 

Great! Lisbon is a fantastic choice with its rich history and delicious food. I'll help you plan a wonderful 2-day trip.

Let's start with **Day 1**. How does this sound?

### Day 1: Historic Alfama & Baixa

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Begin your day in Alfama, one of Lisbon's oldest and most charming neighborhoods. Wander through its narrow, winding streets, soak in the atmosphere, and discover hidden viewpoints.
    *   Visit the **Lisbon Cathedral (Sé de Lisboa)**, a majestic Romanesque cathedral that is the oldest church in the city.
    *   Head up to **São Jorge Castle (Castelo de São Jorge)** for panoramic views of Lisbon and the Tagus River, and explore its historic ramparts and archaeological site.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Cuisine in Alfama**
    *   Enjoy a traditional Portuguese lunch at a local tasca (tavern) in Alfama, savoring dishes like grilled sardines or bacalhau (codfish).
*   **Afternoon (2:30 PM - 6:00 PM): Baixa District & Praça do Comércio**
    *   Descend from Alfama into the Baixa district, rebuilt after the 1755 earthquake. Stroll down **Rua Augusta**, a bustling pedestrian street, and admire the elegant architecture.
    *   Walk through the magnificent **Praça do Comércio**, one of Europe's largest and most beautiful squares, right on the Tagus River.
    *   Consider a ride on the **Santa Justa Lift** for more city views and an architectural marvel (be prepared for queues).
*   **Evening (7:00 PM onwards): Dinner & Fado in Alfama/Bairro Alto**
    *   Enjoy dinner at a restaurant offering traditional Portuguese dishes.
    *   Experience a live **Fado show**, Portugal's melancholic and soulful musical genre, often found in intimate venues in Alfama or the nearby Bairro Alto.

How does this sound for your first day in Lisbon?

--------------------------------------------------


Created a BRAND NEW session for Turn 2: 23c5557b-203a-42ae-aa81-54ea030f74cc
🗣️ User (Turn 2): 'Yes, that looks perfect! Please plan Day 2.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '23c5557b-203a-42ae-aa81-54ea030f74cc'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Great! I'm glad Day 1 looks good. Let's plan out Day 2 for your trip to Kyoto!

Here's a suggestion for your second day, focusing on some more iconic and picturesque areas:

### Day 2: Arashiyama Bamboo Grove & Temples

*   **Morning (9:00 AM - 12:00 PM): Arashiyama Bamboo Grove and Tenryu-ji Temple**
    *   Start your day early with a visit to the iconic Arashiyama Bamboo Grove. Enjoy a serene walk through the towering bamboo stalks.
    *   Immediately adjacent to the grove is Tenryu-ji Temple, a UNESCO World Heritage site known for its beautiful landscape garden. Explore the temple grounds and enjoy

Great! I'm glad Day 1 looks good. Let's plan out Day 2 for your trip to Kyoto!

Here's a suggestion for your second day, focusing on some more iconic and picturesque areas:

### Day 2: Arashiyama Bamboo Grove & Temples

*   **Morning (9:00 AM - 12:00 PM): Arashiyama Bamboo Grove and Tenryu-ji Temple**
    *   Start your day early with a visit to the iconic Arashiyama Bamboo Grove. Enjoy a serene walk through the towering bamboo stalks.
    *   Immediately adjacent to the grove is Tenryu-ji Temple, a UNESCO World Heritage site known for its beautiful landscape garden. Explore the temple grounds and enjoy the tranquility.
*   **Lunch (12:00 PM - 1:30 PM): Local Cuisine in Arashiyama**
    *   Dine at one of the many restaurants in the Arashiyama area. You can find options ranging from traditional Japanese set meals (kaiseki) to more casual noodle shops.
*   **Afternoon (1:30 PM - 5:00 PM): Togetsukyo Bridge & Arashiyama Monkey Park Iwatayama**
    *   Walk across the picturesque Togetsukyo Bridge, offering lovely views of the Hozugawa River and the surrounding mountains.
    *   For a unique experience and panoramic views of Kyoto, visit the Arashiyama Monkey Park Iwatayama. It involves a short hike up, but you'll be able to see wild Japanese macaques up close.
*   **Evening (6:00 PM onwards): Dinner & Relaxation in Gion**
    *   Head back towards the Gion district (or another area you'd like to explore for dinner). Enjoy dinner at a local izakaya or a traditional restaurant.
    *   After dinner, take a leisurely stroll through Gion, hoping to catch a glimpse of geisha or maiko.

How does this sound for Day 2? Would you like any changes or adjustments?

--------------------------------------------------



See? The agent was confused! It likely asked what destination or what trip we were talking about. Because the second query was in a fresh, isolated session, the agent had no memory of planning Day 1 in Lisbon.

This perfectly illustrates why **managing sessions is the key to building truly conversational agents!**

---
## 🎉 Congratulations! 🎉

Congratulations on completing your ADK adventure into Tools and Memory! You've taken a massive leap from building single-shot agents to creating dynamic, stateful AI systems.

Let's recap the powerful concepts you've mastered:

- **Fundamental Agent & Tools**: You started by building a "Day Trip Genie" and equipped it with its first tool, GoogleSearch.

- **Custom Function Tools**: You gave your agent a new sense by creating a custom tool to fetch live data from the U.S. National Weather Service API.

- **Agent-as-a-Tool**: You orchestrated a sophisticated hierarchy where agents delegate tasks to other, more specialized agents, creating a collaborative team.

- **The Power of Memory**: Most importantly, you saw firsthand how managing a single, persistent Session allows an agent to remember context, adapt to user feedback, and conduct a meaningful, multi-turn conversation.

```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
